# GOLD ATP ENTRY TYPE

## Imports

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window
import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [4]:
try:
    spark = SparkSession.builder.appName("dim_entry").getOrCreate()
except Exception as e:
    print(e)

In [6]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [ ]:
tb_player_match = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_player_match")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
)

## Tournaments

In [8]:
df = (
    tb_player_match
    .select("PLAYER_ENTRY")
    .distinct()
    .withColumn(
        "ENTRY_TYPE",
        f.when(f.col("PLAYER_ENTRY") == "WC", f.lit("Wild Card"))
        .when(f.col("PLAYER_ENTRY") == "Q", f.lit("Qualifier"))
        .when(f.col("PLAYER_ENTRY") == "LL", f.lit("Lucky Loser"))
        .when(f.col("PLAYER_ENTRY") == "ITF", f.lit("ITF Entry"))
        .when(f.col("PLAYER_ENTRY") == "UP", f.lit("Next Gen / Unranked Performance"))
        .when(f.col("PLAYER_ENTRY") == "W", f.lit("Wild Card"))
        .when(f.col("PLAYER_ENTRY") == "SE", f.lit("Special Exempt"))
        .when(f.col("PLAYER_ENTRY") == "PR", f.lit("Protected Ranking"))
        .when(f.col("PLAYER_ENTRY") == "S", f.lit("Exempt Special / Special"))
        .when(f.col("PLAYER_ENTRY") == "NG", f.lit("Next Gen Accelerator"))
        .otherwise(f.lit("Direct Acceptance / Regular"))
    )
    .withColumn("SK_ENTRY_TYPE", f.monotonically_increasing_id() + 1)
    .select(
        f.col("SK_ENTRY_TYPE"),
        f.col("PLAYER_ENTRY").alias("ENTRY_TYPE_ID"),
        f.col("ENTRY_TYPE").alias("ENTRY_TYPE_NAME")
    )
)

## Save dataframe

### Local

In [9]:
df.toPandas().to_csv(
    r"../../../data/gold/dimension/dim_entry.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [ ]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.dim_entry")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)